# Portfolio Optimizer Training Notebook

This notebook trains the checkpoint files used by the portfolio optimizer application.

Recommended workflow:
1. Point the notebook at a multi-year local price dataset.
2. Load and validate the data once.
3. Train the continuous-weight checkpoints.
4. Train the graph-policy checkpoints.
5. Run the built-in inference round-trip to confirm each checkpoint loads cleanly.

Notes:
- The `10m`, `60m`, and `1440m` labels refer to the bar cadence after resampling.
- Each training job below uses `horizon_steps = 1`. The forecast horizon is one bar ahead at the resampled cadence.
- For production training, prefer a long local dataset over live Alpaca pulls.


## Runtime Notes

- Default this notebook to CPU unless you have a working CUDA install.
- You can either set Alpaca credentials in the config cell below or use `ALPACA_API_KEY` / `ALPACA_API_SECRET` from the environment.
- If you put credentials directly in the notebook for local use, do not commit them.
- If you use Alpaca, cache the fetched data locally so repeated training runs are deterministic.


In [ ]:
import gc
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from core.alpaca_data import configure_alpaca_credentials, fetch_intraday_bars
from core.portfolio_rl import (
    format_price_frame,
    optimize_portfolio,
    optimize_portfolio_inference,
)

print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
print('project root:', ROOT)


In [ ]:
# -----------------------------
# User configuration
# -----------------------------

# Preferred training path: multi-year local dataset.
DATA_FILE = ROOT / 'data' / 'bulk_prices' / 'ten_year_prices2.csv'
DATA_SOURCE = 'csv' if DATA_FILE.exists() else 'alpaca'  # 'csv' or 'alpaca'
DATE_COL = 'Date'
PRICE_COL = 'Close'
TICKER_COL = 'Ticker'

# Optional Alpaca path for smaller experiments.
ALPACA_TICKERS = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA']
ALPACA_START = pd.Timestamp.today() - pd.DateOffset(months=6)
ALPACA_END = pd.Timestamp.today()
ALPACA_CACHE = ROOT / 'data' / 'bulk_prices' / 'alpaca_intraday_training.parquet'
ALPACA_API_KEY = ''      # Optional: set locally here or leave blank to use the environment.
ALPACA_API_SECRET = ''   # Optional: set locally here or leave blank to use the environment.

# Training specs. The resample rule defines the true forecast horizon.
TRAIN_SPECS = [
    {'label': '10m', 'resample_rule': '10min', 'horizon_steps': 1, 'checkpoint_stem': 'portfolio_policy_10m'},
    {'label': '60m', 'resample_rule': '60min', 'horizon_steps': 1, 'checkpoint_stem': 'portfolio_policy_60m'},
    {'label': '1440m', 'resample_rule': '1D', 'horizon_steps': 1, 'checkpoint_stem': 'portfolio_policy_1440m'},
]

TOP_K = 5
EPISODES = 60
LR = 1e-3
RISK_AVERSION = 0.02
DEVICE = 'cpu'  # Change to 'cuda' only after confirming your CUDA/PyTorch stack is healthy.
CPU_THREADS = 8
RL_HIDDEN_DIM = 64
RL_PATIENCE = 20
RL_MIN_DELTA = 1e-4

FORECAST_PARAMS = {
    'stats': {'SeasonalNaive': {'season_length': 5}},
    'ml': {'rf_params': {'n_estimators': 300, 'max_depth': 8}},
}

MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('data source:', DATA_SOURCE)
print('device:', DEVICE)
print('models dir:', MODELS_DIR)


In [ ]:
def clear_device_memory():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass


def require_alpaca_credentials():
    return configure_alpaca_credentials(
        api_key=ALPACA_API_KEY,
        api_secret=ALPACA_API_SECRET,
        persist_env=True,
    )


def rename_common_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    cols = {c.lower(): c for c in df.columns}

    if DATE_COL not in df.columns:
        for candidate in ['date', 'datetime', 'timestamp', 'ds']:
            if candidate in cols:
                rename_map[cols[candidate]] = DATE_COL
                break

    if PRICE_COL not in df.columns:
        for candidate in ['close', 'adj close', 'adj_close', 'y']:
            if candidate in cols:
                rename_map[cols[candidate]] = PRICE_COL
                break

    if TICKER_COL not in df.columns:
        for candidate in ['ticker', 'symbol', 'unique_id']:
            if candidate in cols:
                rename_map[cols[candidate]] = TICKER_COL
                break

    return df.rename(columns=rename_map)


def load_training_prices() -> pd.DataFrame:
    if DATA_SOURCE == 'alpaca':
        require_alpaca_credentials()
        cache_csv = ALPACA_CACHE.with_suffix('.csv')
        if ALPACA_CACHE.exists():
            raw = pd.read_parquet(ALPACA_CACHE)
            print('Loaded cached Alpaca parquet:', ALPACA_CACHE)
        elif cache_csv.exists():
            raw = pd.read_csv(cache_csv)
            print('Loaded cached Alpaca csv:', cache_csv)
        else:
            raw = fetch_intraday_bars(ALPACA_TICKERS, start=ALPACA_START, end=ALPACA_END)
            if raw.empty:
                raise RuntimeError('Alpaca returned no data for the requested training window.')
            ALPACA_CACHE.parent.mkdir(parents=True, exist_ok=True)
            raw.to_parquet(ALPACA_CACHE, index=False)
            raw.to_csv(cache_csv, index=False)
            print('Fetched and cached Alpaca data.')
    else:
        if not DATA_FILE.exists():
            raise FileNotFoundError(
                f'Local training file not found: {DATA_FILE}. Update DATA_FILE or switch DATA_SOURCE to alpaca.'
            )
        raw = pd.read_csv(DATA_FILE)
        print('Loaded local csv:', DATA_FILE)

    raw = rename_common_columns(raw)
    prices = format_price_frame(raw, date_col=DATE_COL, price_col=PRICE_COL, ticker_col=TICKER_COL)
    if prices.empty:
        raise ValueError('No usable price rows were found after normalization.')
    return prices


def checkpoint_path_for(spec: dict, rl_mode: str) -> Path:
    suffix = '_graph' if rl_mode == 'graph' else ''
    return MODELS_DIR / f"{spec['checkpoint_stem']}{suffix}.pt"


def run_training_job(spec: dict, prices: pd.DataFrame, rl_mode: str = 'weights') -> dict:
    clear_device_memory()
    ckpt = checkpoint_path_for(spec, rl_mode)
    mode_name = 'graph' if rl_mode == 'graph' else 'weights'
    print()
    print(f"=== Training {mode_name} policy for {spec['label']} ({spec['resample_rule']}) ===")
    result = optimize_portfolio(
        prices=prices,
        horizon=spec['horizon_steps'],
        top_k=TOP_K,
        episodes=EPISODES,
        lr=LR,
        risk_aversion=RISK_AVERSION,
        checkpoint_path=str(ckpt),
        resample_rule=spec['resample_rule'],
        device=DEVICE,
        rl_mode=rl_mode,
        forecast_params=FORECAST_PARAMS,
        rl_hidden_dim=RL_HIDDEN_DIM,
        cpu_threads=CPU_THREADS,
        rl_patience=RL_PATIENCE,
        rl_min_delta=RL_MIN_DELTA,
    )
    inference = optimize_portfolio_inference(
        prices=prices,
        checkpoint_path=str(ckpt),
        horizon=spec['horizon_steps'],
        risk_aversion=RISK_AVERSION,
        resample_rule=spec['resample_rule'],
        device=DEVICE,
        rl_mode=rl_mode,
        forecast_params=FORECAST_PARAMS,
        cpu_threads=CPU_THREADS,
    )
    summary = {
        'label': spec['label'],
        'resample_rule': spec['resample_rule'],
        'rl_mode': mode_name,
        'checkpoint': str(ckpt),
        'engine': result['optimizer_engine'],
        'train_expected_horizon_return': result['portfolio_metrics']['expected_horizon_return'],
        'inference_expected_horizon_return': inference['portfolio_metrics']['expected_horizon_return'],
        'n_assets': len(result['portfolio']),
    }
    return {
        'checkpoint': ckpt,
        'train': result,
        'inference': inference,
        'summary': summary,
    }


In [ ]:
prices = load_training_prices()
print(f'Loaded {len(prices):,} rows across {prices["ticker"].nunique()} tickers.')
print('Coverage:', prices['ds'].min(), '->', prices['ds'].max())
display(prices.head())


## Train Continuous-Weight Checkpoints

This loop produces:
- `models/portfolio_policy_10m.pt`
- `models/portfolio_policy_60m.pt`
- `models/portfolio_policy_1440m.pt`


In [ ]:
weight_runs = {}
weight_summaries = []
for spec in TRAIN_SPECS:
    run = run_training_job(spec, prices, rl_mode='weights')
    weight_runs[spec['label']] = run
    weight_summaries.append(run['summary'])

weight_summary = pd.DataFrame(weight_summaries).sort_values(['label'])
display(weight_summary)


In [ ]:
# Inspect the latest continuous-weight inference result.
weight_runs['10m']['inference']['portfolio']


## Train Graph-Policy Checkpoints

This loop produces:
- `models/portfolio_policy_10m_graph.pt`
- `models/portfolio_policy_60m_graph.pt`
- `models/portfolio_policy_1440m_graph.pt`


In [ ]:
graph_runs = {}
graph_summaries = []
for spec in TRAIN_SPECS:
    run = run_training_job(spec, prices, rl_mode='graph')
    graph_runs[spec['label']] = run
    graph_summaries.append(run['summary'])

graph_summary = pd.DataFrame(graph_summaries).sort_values(['label'])
display(graph_summary)


In [ ]:
# Inspect the latest graph-policy inference result.
graph_runs['10m']['inference']['portfolio']


## Save a Combined Training Summary

This gives you a compact record of what was trained and where the checkpoints were written.


In [ ]:
summary_dir = ROOT / 'data' / 'training_runs'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / 'portfolio_training_summary.csv'
combined_summary = pd.concat([weight_summary, graph_summary], ignore_index=True)
combined_summary.to_csv(summary_path, index=False)
print('Saved summary to', summary_path)
display(combined_summary)


## Using the Checkpoints in the App

After training, launch the desktop app and point the Portfolio Optimizer screen at one of the saved checkpoint paths in `models/`.

Recommended mapping:
- `portfolio_policy_10m.pt` for the continuous-weight 10 minute policy
- `portfolio_policy_60m.pt` for the continuous-weight 60 minute policy
- `portfolio_policy_1440m.pt` for the continuous-weight daily policy
- `portfolio_policy_10m_graph.pt` for the graph 10 minute policy
- `portfolio_policy_60m_graph.pt` for the graph 60 minute policy
- `portfolio_policy_1440m_graph.pt` for the graph daily policy
